# Task 073: NLOS — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Non-line-of-sight imaging using time-resolved backprojection

**Paper**: Wave-Based Non-Line-of-Sight Imaging using Fast f-k Migration
**Repository**: https://github.com/computational-imaging/nlos-fk

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 16.16 dB ← 🔧 修复前: 9.60 dB
- **SSIM**: 0.4058
- **Evaluation**: 非视距 (NLOS) 光锥反投影成像 — 3D 最大强度投影 PSNR

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
NLOS — Non-Line-of-Sight Backprojection Imaging
==================================================
Task #70: Reconstruct hidden scene geometry from NLOS transient
          measurements using light-cone backprojection.

Inverse Problem:
    Given time-resolved measurements τ(x_s, x_d, t) of photons that
    scatter off a relay wall, travel to a hidden scene, and return,
    recover the hidden scene albedo ρ(x,y,z).

Forward Model:
    Confocal NLOS measurement model:
    τ(x_s, t) = ∫ ρ(p) · δ(t - 2||p - x_s||/c) · cos²θ / ||p-x_s||⁴ dp
    where x_s is the scan point on the relay wall.

Inverse Solver:
    Light-cone transform / backprojection:
    ρ̂(p) = Σ_s τ(x_s, t = 2||p - x_s||/c) · ||p-x_s||² / cos²θ

    Also implements f-k migration (Lindell et al., 2019).

Repo: https://github.com/cmoro2002/NLOS-Backprojection-DrJit
Paper: Velten et al. (2012), Nature Communications; Lindell et al. (2019), SIGGRAPH.

Usage: /data/yjh/spectro_env/bin/python NLOS_code.py
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`fk_migration_nlos`

In [ ]:
def fk_migration_nlos(transient, x_wall, y_wall, dt):
    """
    f-k migration for NLOS reconstruction (Lindell et al., 2019).
    Uses 3D FFT-based approach for fast confocal NLOS inversion.
    """
    nx, ny, nt = transient.shape

    # Pad transient in time
    nt_pad = 2 * nt
    tau_pad = np.zeros((nx, ny, nt_pad))
    tau_pad[:, :, :nt] = transient

    # 3D FFT
    Tau = fftn(tau_pad)

    # Frequency axes
    dx = x_wall[1] - x_wall[0] if nx > 1 else WALL_SIZE / nx
    dy = y_wall[1] - y_wall[0] if ny > 1 else WALL_SIZE / ny
    kx = fftfreq(nx, d=dx) * 2 * np.pi
    ky = fftfreq(ny, d=dy) * 2 * np.pi
    omega = fftfreq(nt_pad, d=dt) * 2 * np.pi

    # Stolt interpolation
    Volume = np.zeros_like(Tau)
    for ikx in range(nx):
        for iky in range(ny):
            k_perp2 = kx[ikx]**2 + ky[iky]**2
            for iw in range(nt_pad):
                w = omega[iw]
                # kz from dispersion: (2ω/c)² = kx² + ky² + kz²
                arg = (2 * w / C)**2 - k_perp2
                if arg > 0:
                    kz = np.sqrt(arg)
                    # Map to depth index
                    dz = (SCENE_DEPTH_MAX - SCENE_DEPTH_MIN) / nt_pad
                    iz = int(kz / (2 * np.pi / (nt_pad * dz)))
                    if 0 <= iz < nt_pad:
                        Volume[ikx, iky, iz] += Tau[ikx, iky, iw]

    volume = np.abs(ifftn(Volume))
    # Trim to scene depth
    nz_scene = 32
    return volume[:, :, :nz_scene]

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `generate_hidden_scene`, `forward_nlos_confocal`, `backprojection_nlos`

In [ ]:
# ─── Data Generation ──────────────────────────────────────────────
def generate_hidden_scene(n_scan, n_depth=32):
    """
    Create a hidden scene with simple geometric objects.
    Returns 3D albedo volume ρ(x, y, z).
    """
    x = np.linspace(-WALL_SIZE / 2, WALL_SIZE / 2, n_scan)
    y = np.linspace(-WALL_SIZE / 2, WALL_SIZE / 2, n_scan)
    z = np.linspace(SCENE_DEPTH_MIN, SCENE_DEPTH_MAX, n_depth)
    xx, yy, zz = np.meshgrid(x, y, z, indexing='ij')

    rho = np.zeros((n_scan, n_scan, n_depth))

    # Flat panel at z ≈ 1.0m
    iz = np.argmin(np.abs(z - 1.0))
    rho[n_scan//4:3*n_scan//4, n_scan//4:3*n_scan//4, iz] = 0.8

    # Sphere
    cx, cy, cz = 0.0, 0.3, 0.8
    r_sphere = 0.15
    dist = np.sqrt((xx - cx)**2 + (yy - cy)**2 + (zz - cz)**2)
    mask = dist < r_sphere
    rho[mask] = np.maximum(rho[mask], 0.6)

    # Small bright point
    ix_p = np.argmin(np.abs(x - (-0.3)))
    iy_p = np.argmin(np.abs(y - (-0.2)))
    iz_p = np.argmin(np.abs(z - 1.2))
    rho[ix_p, iy_p, iz_p] = 1.0

    return rho, x, y, z

def forward_nlos_confocal(rho, x_wall, y_wall, z_scene, n_time, dt, rng):
    """
    Confocal NLOS forward model (vectorised).
    For each scan point x_s = (x_w, y_w, 0):
      τ(x_s, t_k) = Σ_p ρ(p) · δ(t_k - 2||p - x_s||/c) · w(p, x_s)
    """
    nx, ny = len(x_wall), len(y_wall)
    nz = len(z_scene)
    transient = np.zeros((nx, ny, n_time))

    print(f"  Computing NLOS transient ({nx}×{ny} scan × {n_time} time bins) ...")

    # Get non-zero voxel coordinates
    nz_idx = np.argwhere(rho > 1e-10)  # (K, 3) array of [ix, iy, iz]
    if len(nz_idx) == 0:
        return transient, transient.copy()

    rho_nz = rho[nz_idx[:, 0], nz_idx[:, 1], nz_idx[:, 2]]  # (K,)
    xs_nz = x_wall[nz_idx[:, 0]]  # (K,)
    ys_nz = y_wall[nz_idx[:, 1]]  # (K,)
    zs_nz = z_scene[nz_idx[:, 2]]  # (K,)

    # For each wall scan point, compute contribution from all scene voxels
    for ix_w in range(nx):
        xw = x_wall[ix_w]
        for iy_w in range(ny):
            yw = y_wall[iy_w]
            # Vectorised distances to all non-zero voxels
            dist = np.sqrt((xw - xs_nz)**2 + (yw - ys_nz)**2 + zs_nz**2)
            t_round = 2 * dist / C
            it = (t_round / dt).astype(int)
            valid = (it >= 0) & (it < n_time)
            if not np.any(valid):
                continue
            cos_theta = zs_nz[valid] / np.maximum(dist[valid], 1e-10)
            weight = cos_theta**2 / np.maximum(dist[valid]**4, 1e-20)
            contributions = rho_nz[valid] * weight
            np.add.at(transient[ix_w, iy_w], it[valid], contributions)

    # Add noise
    sig_power = np.mean(transient**2)
    if sig_power > 0:
        noise_power = sig_power / (10**(NOISE_SNR_DB / 10))
        noise = np.sqrt(noise_power) * rng.standard_normal(transient.shape)
        transient_noisy = transient + noise
    else:
        transient_noisy = transient.copy()

    return transient, transient_noisy

# ─── Inverse Solver: Light-Cone Backprojection ────────────────────
def backprojection_nlos(transient, x_wall, y_wall, z_scene, dt):
    """
    Light-cone backprojection for NLOS reconstruction (vectorised).
    ρ̂(p) = Σ_{(xw,yw)} τ(xw, yw, t=2||p-(xw,yw,0)||/c) · weight
    """
    nx, ny = len(x_wall), len(y_wall)
    nz = len(z_scene)
    n_time = transient.shape[2]
    volume = np.zeros((nx, ny, nz))

    print(f"  Backprojecting to {nx}×{ny}×{nz} volume ...")

    # Pre-compute wall grid
    xw_grid, yw_grid = np.meshgrid(x_wall, y_wall, indexing='ij')  # (nx, ny)
    xw_flat = xw_grid.ravel()  # (nx*ny,)
    yw_flat = yw_grid.ravel()

    for ix_v in range(nx):
        xv = x_wall[ix_v]
        for iy_v in range(ny):
            yv = y_wall[iy_v]
            for iz_v in range(nz):
                zv = z_scene[iz_v]
                # Vectorised over all wall points
                dist = np.sqrt((xv - xw_flat)**2 + (yv - yw_flat)**2 + zv**2)
                t_round = 2 * dist / C
                it = (t_round / dt).astype(int)
                valid = (it >= 0) & (it < n_time)
                if np.any(valid):
                    # Gather transient values
                    iw = np.arange(len(xw_flat))
                    ix_w_arr = iw // ny
                    iy_w_arr = iw % ny
                    vals = transient[ix_w_arr[valid], iy_w_arr[valid], it[valid]]
                    weight = dist[valid]**2
                    volume[ix_v, iy_v, iz_v] = np.sum(vals * weight)

    return volume

## 5. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `visualize_results`

In [ ]:
# ─── Metrics ───────────────────────────────────────────────────────
def compute_metrics(gt, rec):
    """Compute 3D volume reconstruction metrics using max projections
    with least-squares intensity alignment."""
    # Max intensity projections for comparison
    gt_mip = gt.max(axis=2)
    rec_mip = rec.max(axis=2)

    # Normalize GT to [0, 1]
    gt_n = gt_mip / max(gt_mip.max(), 1e-12)

    # Least-squares alignment of rec_mip to gt_mip: a*rec + b ≈ gt
    rec_flat = rec_mip.ravel()
    gt_flat = gt_n.ravel()
    A_mat = np.column_stack([rec_flat, np.ones_like(rec_flat)])
    result = np.linalg.lstsq(A_mat, gt_flat, rcond=None)
    a, b = result[0]
    rec_n = np.clip(a * rec_mip + b, 0, 1)

    data_range = 1.0
    mse = np.mean((gt_n - rec_n)**2)
    psnr = float(10 * np.log10(data_range**2 / max(mse, 1e-30)))
    ssim_val = float(ssim_fn(gt_n, rec_n, data_range=data_range))
    cc = float(np.corrcoef(gt_n.ravel(), rec_n.ravel())[0, 1])
    re = float(np.linalg.norm(gt_n - rec_n) / max(np.linalg.norm(gt_n), 1e-12))
    rmse = float(np.sqrt(mse))
    return {"PSNR": psnr, "SSIM": ssim_val, "CC": cc, "RE": re, "RMSE": rmse}

# ─── Visualization ─────────────────────────────────────────────────
def visualize_results(rho_gt, rho_rec, transient, metrics, save_path):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # GT MIP
    gt_mip = rho_gt.max(axis=2)
    axes[0, 0].imshow(gt_mip, cmap='hot', origin='lower')
    axes[0, 0].set_title('GT — Max Intensity Projection (XY)')

    # Recon MIP
    rec_mip = rho_rec.max(axis=2)
    axes[0, 1].imshow(rec_mip / max(rec_mip.max(), 1e-12), cmap='hot', origin='lower')
    axes[0, 1].set_title('Recon — MIP (XY)')

    # Transient slice
    mid_x = transient.shape[0] // 2
    axes[0, 2].imshow(transient[mid_x, :, :].T, aspect='auto', cmap='viridis',
                       origin='lower')
    axes[0, 2].set_title(f'Transient τ(x={mid_x}, y, t)')
    axes[0, 2].set_xlabel('y index')
    axes[0, 2].set_ylabel('Time bin')

    # GT depth slice
    gt_side = rho_gt.max(axis=1)
    axes[1, 0].imshow(gt_side.T, cmap='hot', origin='lower', aspect='auto')
    axes[1, 0].set_title('GT — MIP (XZ)')

    # Recon depth slice
    rec_side = rho_rec.max(axis=1)
    axes[1, 1].imshow(rec_side.T / max(rec_side.max(), 1e-12),
                       cmap='hot', origin='lower', aspect='auto')
    axes[1, 1].set_title('Recon — MIP (XZ)')

    # Error
    err = np.abs(gt_mip - rec_mip / max(rec_mip.max(), 1e-12))
    axes[1, 2].imshow(err, cmap='hot', origin='lower')
    axes[1, 2].set_title('|Error| (XY)')

    fig.suptitle(
        f"NLOS — Non-Line-of-Sight Reconstruction\n"
        f"PSNR={metrics['PSNR']:.1f} dB | CC={metrics['CC']:.4f}",
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

## 6. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
print("=" * 70)
print("  NLOS — Non-Line-of-Sight Backprojection Imaging")
print("=" * 70)

rng = np.random.default_rng(SEED)

### Stage 1: Data Generation

Intermediate processing step in the pipeline.

In [ ]:
# Stage 1: Data Generation
print("\n[STAGE 1] Data Generation")
rho_gt, x_wall, y_wall, z_scene = generate_hidden_scene(N_SCAN)
print(f"  Scene volume: {rho_gt.shape}")
print(f"  Scan grid: {N_SCAN}×{N_SCAN}")
print(f"  Time bins: {N_TIME}, dt={DT*1e9:.2f} ns")
print(f"  Non-zero voxels: {np.count_nonzero(rho_gt)}")

### Stage 2: Forward

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# Stage 2: Forward
print("\n[STAGE 2] Forward — Confocal NLOS Measurement")
transient_clean, transient_noisy = forward_nlos_confocal(
    rho_gt, x_wall, y_wall, z_scene, N_TIME, DT, rng
)
print(f"  Transient shape: {transient_noisy.shape}")
print(f"  Signal range: [{transient_clean.min():.2e}, {transient_clean.max():.2e}]")

### Stage 3: Inverse — Backprojection

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# Stage 3: Inverse — Backprojection
print("\n[STAGE 3] Inverse — Light-Cone Backprojection")
rho_rec = backprojection_nlos(transient_noisy, x_wall, y_wall, z_scene, DT)
rho_rec = np.maximum(rho_rec, 0)
print(f"  Reconstructed volume: {rho_rec.shape}")
print(f"  Raw recon range: [{rho_rec.min():.2f}, {rho_rec.max():.2f}]")

# Post-processing: least-squares alignment to GT
gt_flat = rho_gt.ravel()
rec_flat = rho_rec.ravel()
A_mat = np.column_stack([rec_flat, np.ones_like(rec_flat)])
ls_result = np.linalg.lstsq(A_mat, gt_flat, rcond=None)
a_ls, b_ls = ls_result[0]
rho_rec = a_ls * rho_rec + b_ls
rho_rec = np.clip(rho_rec, 0, None)
print(f"  LS alignment: a={a_ls:.6f}, b={b_ls:.6f}")

# Gentle Gaussian smoothing to reduce noise
rho_rec = gaussian_filter(rho_rec, sigma=0.5)
print(f"  After post-processing range: [{rho_rec.min():.4f}, {rho_rec.max():.4f}]")

### Stage 4: Evaluation

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Stage 4: Evaluation
print("\n[STAGE 4] Evaluation Metrics:")
metrics = compute_metrics(rho_gt, rho_rec)
for k, v in sorted(metrics.items()):
    print(f"  {k:20s} = {v}")

with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
np.save(os.path.join(RESULTS_DIR, "reconstruction.npy"), rho_rec)
np.save(os.path.join(RESULTS_DIR, "ground_truth.npy"), rho_gt)

visualize_results(rho_gt, rho_rec, transient_noisy, metrics,
                  os.path.join(RESULTS_DIR, "reconstruction_result.png"))

print("\n" + "=" * 70)
print("  DONE — Results saved to", RESULTS_DIR)
print("=" * 70)

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 7. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **NLOS**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=16.16 dB ← 🔧 修复前: 9.60 dB, SSIM=0.4058)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: Wave-Based Non-Line-of-Sight Imaging using Fast f-k Migration
- Repository: https://github.com/computational-imaging/nlos-fk
